In [672]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import emcee
from find_source import summary
import corner
import math
from astropy.coordinates import Angle
import astropy.units as units
import scipy.special as sp
import warnings

In [ ]:
def p_log_likelihood(params, re, im, u, v, w):
    i0, l0, m0 = params
    model = i0 * np.exp(-2*np.pi*1j*(u*l0 + v*m0))
    return -0.5 * np.sum(w * ((re - model.real)**2 + (im - model.imag)**2))

def p_log_prior(params, rad_coord, rad_bmaj):
    i0, l0, m0 = params
    l0_off = abs(l0 - rad_coord[0])
    m0_off = abs(m0 - rad_coord[1])
    if 0 < i0 and l0_off < 2*rad_bmaj and m0_off < 2*rad_bmaj:
        return 0.0
    return -np.inf

def p_log_probability(params, re, im, u, v, w, rad_coord, rad_bmaj):
    lp = p_log_prior(params, rad_coord, rad_bmaj)
    if not np.isfinite(lp): # if parameter(s) outside of our range
        return -np.inf # essentially 0 probability (e^-inf = 0)
    return lp + p_log_likelihood(params, re, im, u, v, w)

def cg_log_likelihood(params, re, im, u, v, w):
    s0, l0, m0, sigma = params # Jy, rad, rad, rad
    model = s0 * np.exp(-0.5 * (u**2/sigma**2 + v**2/sigma**2)) * np.exp(-2*np.pi*1j*(u*l0 + v*m0))
    return -0.5 * np.sum(w * ((re - model.real)**2 + (im - model.imag)**2))

def cg_log_prior(params, rad_coord0, rad_bmaj):
    s0, l0, m0, sigma = params
    l0_off = abs(l0 - rad_coord0[0])
    m0_off = abs(m0 - rad_coord0[1])
    if 0 < s0 and 0 < sigma: #and l0_off < 2*rad_bmaj and m0_off < 2*rad_bmaj:
        return 0.0
    return -np.inf

def cg_log_probability(params, re, im, u, v, w, rad_coord0, rad_bmaj):
    lp = cg_log_prior(params, rad_coord0, rad_bmaj)
    if not np.isfinite(lp): # if parameter(s) outside of our range
        return -np.inf # essentially 0 probability (e^-inf = 0)
    return lp + cg_log_likelihood(params, re, im, u, v, w)

def g_log_likelihood(params, re, im, u, v, w):
    s0, l0, m0, sigma_u, ratio, thetav = params
    model = s0 * np.exp(-1/2 * ((u*np.cos(thetav)-v*np.sin(thetav))**2/sigma_u**2 + \
                        (u*np.sin(thetav)+v*np.cos(thetav))**2/(sigma_u*ratio)**2)) \
                        * np.exp(-2*np.pi*1j*(u*l0 + v*m0))
    return -0.5 * np.sum(w * ((re - model.real)**2 + (im - model.imag)**2))

def g_log_prior(params, rad_coord, rad_bmaj):
    s0, l0, m0, sigma_u, ratio, thetav = params
    l0_off = np.abs(l0 - rad_coord[0])
    m0_off = np.abs(m0 - rad_coord[1])
    if 0 < s0 and 0 < sigma_u and 0 < ratio <= 1 and l0_off < 2*rad_bmaj \
        and m0_off < 2*rad_bmaj and -np.pi/2 <= thetav <= np.pi/2:
        return 0.0
    return -np.inf

def g_log_probability(params, re, im, u, v, w, rad_coord, rad_bmaj):
    lp = g_log_prior(params, rad_coord, rad_bmaj)
    if not np.isfinite(lp): # if parameter(s) outside of our range
        return -np.inf # essentially 0 probability (e^-inf = 0)
    return lp + g_log_likelihood(params, re, im, u, v, w)

def d_log_likelihood(params, re, im, u, v, w):
    s0, l0, m0, r = params
    q = (u**2 + v**2)**0.5 # in lambda
    model = s0 * 2*r/q * sp.j1(q/r) * np.exp(-2*np.pi*1j*(u*l0 + v*m0))
    return -0.5 * np.sum(w * ((re - model.real)**2 + (im - model.imag)**2))

def d_log_prior(params, rad_coord, rad_bmaj):
    s0, l0, m0, r = params
    l0_off = abs(l0 - rad_coord[0])
    m0_off = abs(m0 - rad_coord[1])
    if 0 < s0 and 0 < r: # and l0_off < 2*rad_bmaj and m0_off < 2*rad_bmaj:
        return 0.0
    return -np.inf

def d_log_probability(params, re, im, u, v, w, rad_coord, rad_bmaj):
    lp = d_log_prior(params, rad_coord, rad_bmaj)
    if not np.isfinite(lp): # if parameter(s) outside of our range
        return -np.inf # essentially 0 probability (e^-inf = 0)
    return lp + d_log_likelihood(params, re, im, u, v, w)

def dp_log_likelihood(params, re, im, u, v, w):
    s1, l1, m1, s2, l2, m2 = params
    model = s1 * np.exp(-2*np.pi*1j*(u*l1 + v*m1)) + s2 * np.exp(-2*np.pi*1j*(u*l2 + v*m2))
    return -0.5 * np.sum(w * ((re - model.real)**2 + (im - model.imag)**2))

def dp_log_prior(params, rad_coord1, rad_coord2, rad_bmaj):
    s1, l1, m1, s2, l2, m2 = params
    l1_off = abs(l1 - rad_coord1[0])
    m1_off = abs(m1 - rad_coord1[1])
    l2_off = abs(l2 - rad_coord2[0])
    m2_off = abs(m2 - rad_coord2[1])
    if 0 < s1 and l1_off < 2*rad_bmaj and m1_off < 2*rad_bmaj and 0 < s2 and l2_off < 2*rad_bmaj and m2_off < 2*rad_bmaj: #and s1 >= s2:
        return 0.0
    return -np.inf

def dp_log_probability(params, re, im, u, v, w, rad_coord1, rad_coord2, rad_bmaj):
    lp = dp_log_prior(params, rad_coord1, rad_coord2, rad_bmaj)
    if not np.isfinite(lp): # if parameter(s) outside of our range
        return -np.inf # essentially 0 probability (e^-inf = 0)
    return lp + dp_log_likelihood(params, re, im, u, v, w)

In [ ]:
def point_uv_fit(fits_file: str, re, im, u, v, w, cdelt1, cunit1, corner_plot: bool = True):

    nsteps = 100
    ndim = 3
    nwalkers = 2 * ndim

    # initial positions for re and im
    p0 = np.zeros((nwalkers, ndim))

    rad_pix = float(Angle(cdelt1, cunit1).to(units.radian).value)
    summ = summary(fits_file, plot=False)
    coord0 = summ['int_peak_coord'][0] # relative arcsec
    # coord0 = (0,0) # testing purposes
    rad_coord = (float(Angle(coord0[0], units.arcsec).to(units.radian).value), float(Angle(coord0[1], units.arcsec).to(units.radian).value))
    bmaj = summ['fwhm'] # arcsec
    # bmaj = 3 # arcsec, testing purposes
    rad_bmaj = float(Angle(bmaj, units.arcsec).to(units.radian).value)
    peak_guess = summ['int_peak_val'][0] # Jy
    # peak_guess = 0.01 # testing purposes
    for i in range(nwalkers):
        p0[i,0] = np.random.uniform(0.95*peak_guess, 1.05*peak_guess)
        p0[i,1] = np.random.uniform(-rad_pix/2+rad_coord[0], rad_pix/2+rad_coord[0])
        p0[i,2] = np.random.uniform(-rad_pix/2+rad_coord[1], rad_pix/2+rad_coord[1])

    print(p0) # debug

    sampler = emcee.EnsembleSampler(nwalkers, ndim, p_log_probability, args=(re, im, u, v, w, rad_coord, rad_bmaj))
    try:
        state = sampler.run_mcmc(p0, nsteps)
    except emcee.autocorr.AutocorrError:
        pass
    tau = sampler.get_autocorr_time(quiet=True)
    int_tau = math.ceil(np.nanmax(tau)/1)
    steps_to_200_tau = int_tau * 200 - nsteps
    sampler.run_mcmc(state,  steps_to_200_tau)
    chain = sampler.get_chain(discard = int_tau * 5, flat=True)

    s0_samples = [i[0] for i in chain]
    l0_samples = [i[1] for i in chain]
    m0_samples = [i[2] for i in chain]

    s0_med = float(np.median(s0_samples))
    s0_sd = float(np.nanstd(s0_samples))
    l0_med = float(np.median(l0_samples))
    l0_sd = float(np.nanstd(l0_samples))
    m0_med = float(np.median(m0_samples))
    m0_sd = float(np.nanstd(m0_samples))

    model = s0_med * np.exp(-2*np.pi*1j*(u*l0_med + v*m0_med))
    chi2 = float(np.sum(w * ((re - model.real)**2 + (im - model.imag)**2)))

    if corner_plot:
        fig = corner.corner(chain, labels=["i0", "l0", "m0"])

    return {"peak_med": s0_med, "peak_sd": s0_sd, "x_med": float(Angle(l0_med, 'rad').to(units.arcsec).value), \
            "x_sd": float(Angle(l0_sd, 'rad').to(units.arcsec).value), "y_med": float(Angle(m0_med, 'rad').to(units.arcsec).value), \
            "y_sd": float(Angle(m0_sd, 'rad').to(units.arcsec).value), "chi2": chi2}

In [ ]:
def circlular_gaussian_uv_fit(fits_file: str, re, im, u, v, w, cdelt1, cunit1, naxis1, corner_plot: bool = True):

    nsteps = 100
    ndim = 4
    nwalkers = 2 * ndim

    p0 = np.zeros((nwalkers, ndim))
    # summ = summary(fits_file, plot=False)
    # coord0 = summ['int_peak_coord'][0] # relative arcsec
    coord0 = (-1.0, -1.0) # testing purposes
    rad_coord = (float(Angle(coord0[0], units.arcsec).to(units.radian).value), float(Angle(coord0[1], units.arcsec).to(units.radian).value))
    # bmaj = summ['fwhm'] # arcsec
    # bmin = file[0].header['BMIN'] # arcsec
    bmaj = 3.3 # arcsec, testing purposes
    bmin = 3 # arcsec, testing purposes
    rad_bmaj = Angle(bmaj, 'arcsec').to(units.radian).value
    rad_bmin = Angle(bmin, 'arcsec').to(units.radian).value
    rad_barea = np.pi * rad_bmaj * rad_bmin / (4 * np.log(2)) # because FWHM
    # img_peak = summ['int_peak_val'][0] # Jy
    img_peak = 2.4 # Jy, testing purposes
    total_flux = np.max((re**2 + im**2)**(1/2))
    flux_guess = total_flux # testing purposes
    rad_pix = float(Angle(cdelt1, cunit1).to(units.radian).value)
    l0_guess = rad_coord[0]
    m0_guess = rad_coord[1]
    source_area_guess = total_flux / img_peak * rad_barea
    img_sigma_guess = np.sqrt(source_area_guess / (2*np.pi))
    sigma_guess = 1/(2*np.pi*img_sigma_guess) # lambda

    for i in range(nwalkers):
        p0[i,0] = np.random.uniform(flux_guess*0.95, flux_guess*1.05)
        p0[i,1] = np.random.uniform(-rad_pix/2 + l0_guess, rad_pix/2 + l0_guess)
        p0[i,2] = np.random.uniform(-rad_pix/2 + m0_guess, rad_pix/2 + m0_guess)
        p0[i,3] = np.random.uniform(0.95*sigma_guess, 1.05*sigma_guess)

    sampler = emcee.EnsembleSampler(nwalkers, ndim, cg_log_probability, args=(re, im, u, v, w, rad_coord, rad_bmaj))
    try:
        state = sampler.run_mcmc(p0, nsteps)
    except emcee.autocorr.AutocorrError:
        pass
    tau = sampler.get_autocorr_time(quiet=True)
    int_tau = math.ceil(np.nanmax(tau)/1)
    steps_to_200_tau = int_tau * 200 - nsteps
    sampler.run_mcmc(state,  steps_to_200_tau)
    chain = sampler.get_chain(discard = int_tau * 10, flat=True)

    s0_samples = [i[0] for i in chain]
    l0_samples = [i[1] for i in chain]
    m0_samples = [i[2] for i in chain]
    sigma_samples = [i[3] for i in chain]

    s0_med = float(np.median(s0_samples))
    s0_sd = float(np.nanstd(s0_samples))
    l0_med = float(np.median(l0_samples))
    l0_sd = float(np.nanstd(l0_samples))
    m0_med = float(np.median(m0_samples))
    m0_sd = float(np.nanstd(m0_samples))
    sigma_med = float(np.median(sigma_samples))
    sigma_sd = float(np.nanstd(sigma_samples))

    model = s0_med * np.exp(-0.5 * (u**2/sigma_med**2 + v**2/sigma_med**2)) * np.exp(-2*np.pi*1j*(u*l0_med + v*m0_med))
    chi2 = float(np.sum(w * ((re - model.real)**2 + (im - model.imag)**2)))

    rad_sigma_med_img = 1/(2*np.pi*sigma_med)
    sigma_med_img = Angle(rad_sigma_med_img, units.radian).to(units.arcsec).value
    rad_sigma_sd_img = 1/(2*np.pi*sigma_med**2) * sigma_sd
    sigma_sd_img = Angle(rad_sigma_sd_img, units.radian).to(units.arcsec).value

    source_nbeams = 2*np.pi * rad_sigma_med_img**2 / rad_barea
    peak_med = s0_med / source_nbeams
    peak_sd = s0_sd / source_nbeams

    fig = corner.corner(chain, labels=["s0", "l0", "m0", "sigma"])

    return {"peak_med": float(peak_med), "peak_sd": float(peak_sd), "l0_med": float(Angle(l0_med, units.radian).to(units.arcsec).value), \
            "x_sd": float(Angle(l0_sd, units.radian).to(units.arcsec).value), "y_med": float(Angle(m0_med, units.radian).to(units.arcsec).value), \
            "y_sd": float(Angle(m0_sd, units.radian).to(units.arcsec).value), "sigma_med": float(sigma_med_img), "sigma_sd": float(sigma_sd_img), \
            "chi2": chi2}

In [ ]:
def gaussian_uv_fit(fits_file: str, re, im, u, v, w, cdelt1, cunit1, naxis1, corner_plot: bool = True):

    nsteps = 100
    ndim = 6
    nwalkers = 2 * ndim

    p0 = np.zeros((nwalkers, ndim))
    # summ = summary(fits_file, plot=False)
    # coord0 = summ['int_peak_coord'][0] # relative arcsec
    coord0 = (-1.0, -1.0) # testing purposes
    rad_coord = (float(Angle(coord0[0], units.arcsec).to(units.radian).value), float(Angle(coord0[1], units.arcsec).to(units.radian).value))
    # bmaj = summ['fwhm'] # arcsec
    # bmin = file[0].header['BMIN'] # arcsec
    bmaj = 3.3 # arcsec, testing purposes
    bmin = 3 # arcsec, testing purposes
    rad_bmaj = Angle(bmaj, 'arcsec').to(units.radian).value
    rad_bmin = Angle(bmin, 'arcsec').to(units.radian).value
    rad_barea = np.pi * rad_bmaj * rad_bmin / (4 * np.log(2)) # because FWHM
    # img_peak = summ['int_peak_val'][0] # Jy
    img_peak = 2.4 # Jy, testing purposes
    total_flux = np.max((re**2 + im**2)**(1/2))
    flux_guess = total_flux # testing purposes
    rad_pix = float(Angle(cdelt1, cunit1).to(units.radian).value)
    l0_guess = rad_coord[0]
    m0_guess = rad_coord[1]
    source_area_guess = total_flux / img_peak * rad_barea
    img_sigma_guess = np.sqrt(source_area_guess / (2*np.pi))
    sigma_guess = 1/(2*np.pi*img_sigma_guess) # lambda

    print(Angle(img_sigma_guess, units.radian).to(units.arcsec).value)

    for i in range(nwalkers):
        p0[i,0] = np.random.uniform(flux_guess*0.95, flux_guess*1.05)
        p0[i,1] = np.random.uniform(-rad_pix/2 + l0_guess, rad_pix/2 + l0_guess)
        p0[i,2] = np.random.uniform(-rad_pix/2 + m0_guess, rad_pix/2 + m0_guess)
        p0[i,3] = np.random.uniform(sigma_guess*0.95, sigma_guess*1.05)
        p0[i,4] = np.random.uniform(0, 1)
        p0[i,5] = np.random.uniform(-np.pi/2, np.pi/2)

    print(p0) # debug

    sampler = emcee.EnsembleSampler(nwalkers, ndim, g_log_probability, args=(re, im, u, v, w, rad_coord, rad_bmaj))
    try:
        state = sampler.run_mcmc(p0, nsteps)
    except emcee.autocorr.AutocorrError:
        pass
    tau = sampler.get_autocorr_time(quiet=True)
    int_tau = math.ceil(np.nanmax(tau)/1)
    steps_to_200_tau = int_tau * 200 - nsteps
    sampler.run_mcmc(state,  steps_to_200_tau)
    chain = sampler.get_chain(discard = int_tau * 10, flat=True)

    s0_samples = [i[0] for i in chain]
    l0_samples = [i[1] for i in chain]
    m0_samples = [i[2] for i in chain]
    sigma_u_samples = [i[3] for i in chain]
    ratio_samples = [i[4] for i in chain]
    thetav_samples = [i[5] for i in chain]

    s0_med = float(np.median(s0_samples))
    s0_sd = float(np.nanstd(s0_samples))
    l0_med = float(np.median(l0_samples))
    l0_sd = float(np.nanstd(l0_samples))
    m0_med = float(np.median(m0_samples))
    m0_sd = float(np.nanstd(m0_samples))
    sigma_u_med = float(np.median(sigma_u_samples))
    sigma_u_sd = float(np.nanstd(sigma_u_samples))
    ratio_med = float(np.median(ratio_samples))
    ratio_sd = float(np.nanstd(ratio_samples))
    thetav_med = float(np.median(thetav_samples))
    thetav_sd = float(np.nanstd(thetav_samples))

    rad_sigma_med_img = 1/(2*np.pi*sigma_u_med)
    sigma_med_img = Angle(rad_sigma_med_img, units.radian).to(units.arcsec).value
    rad_sigma_sd_img = 1/(2*np.pi*sigma_u_med**2) * sigma_u_sd
    sigma_sd_img = Angle(rad_sigma_sd_img, units.radian).to(units.arcsec).value

    source_nbeams = 2*np.pi * rad_sigma_med_img**2 / rad_barea
    peak_med = s0_med / source_nbeams
    peak_sd = s0_sd / source_nbeams

    if corner_plot:
        fig = corner.corner(chain, labels=["s0", "l0", "m0", "sigma_min", "ratio", "theta"])

    first_run = {"peak_med": float(peak_med), "peak_sd": float(peak_sd), "x_med": float(Angle(l0_med, units.radian).to(units.arcsec).value), \
            "x_sd": float(Angle(l0_sd, units.radian).to(units.arcsec).value), "y_med": float(Angle(m0_med, units.radian).to(units.arcsec).value), \
            "y_sd": float(Angle(m0_sd, units.radian).to(units.arcsec).value), "sigma_med": float(sigma_med_img), "sigma_sd": float(sigma_sd_img), \
            "ratio_med": ratio_med, "ratio_sd": ratio_sd, "theta_med": float(thetav_med*180/np.pi), \
            "theta_sd": float(thetav_sd*180/np.pi)}

    if first_run['theta_sd'] > 10: # degrees
        if ratio_med > 0.85:
            circular_run = circlular_gaussian_uv_fit(fits_file, re, im, u, v, w, cdelt1, cunit1, naxis1, corner_plot)
            return circular_run

        # it might be getting confused between -90 and 90
        neg_thetav_sample = [i for i in thetav_samples if i < 0]
        pos_thetav_sample = [i for i in thetav_samples if i >= 0]
        neg_med = float(np.median(neg_thetav_sample))
        pos_med = float(np.median(pos_thetav_sample))
        if abs((abs(neg_med) - pos_med)) < 10:
            theta_guess = -np.pi/2
        else:
            theta_guess = thetav_med * np.pi/180

        p1 = np.zeros((nwalkers, ndim))
        for i in range(nwalkers):
            p1[i,0] = np.random.uniform(s0_med*0.95, s0_med*1.05)
            p1[i,1] = np.random.uniform(-rad_pix/4 + l0_guess, rad_pix/4 + l0_guess)
            p1[i,2] = np.random.uniform(-rad_pix/4 + m0_guess, rad_pix/4 + m0_guess)
            p1[i,3] = np.random.uniform(sigma_u_med*0.95, sigma_u_med*1.05)
            p1[i,4] = np.random.uniform(ratio_med*0.95, min(ratio_med*1.05, 1))
            p1[i,5] = np.random.uniform(max(theta_guess - np.pi/36, -np.pi/2), theta_guess + np.pi/36)

        sampler1 = emcee.EnsembleSampler(nwalkers, ndim, g_log_probability, args=(re, im, u, v, w, rad_coord, rad_bmaj))
        nsteps = 100 # update nsteps for second run
        try:
            state = sampler1.run_mcmc(p1, nsteps)
        except emcee.autocorr.AutocorrError:
            pass
        tau = sampler1.get_autocorr_time(quiet=True)

        int_tau = math.ceil(np.nanmax(tau)/1)
        steps_to_200_tau = int_tau * 200 - nsteps
        sampler1.run_mcmc(state,  steps_to_200_tau)
        chain1 = sampler1.get_chain(discard = int_tau * 10, flat=True)

        s0_samples = [i[0] for i in chain1]
        l0_samples = [i[1] for i in chain1]
        m0_samples = [i[2] for i in chain1]
        sigma_u_samples = [i[3] for i in chain1]
        ratio_samples = [i[4] for i in chain1]
        thetav_samples = [i[5] for i in chain1]

        s0_med = float(np.median(s0_samples))
        s0_sd = float(np.nanstd(s0_samples))
        l0_med = float(np.median(l0_samples))
        l0_sd = float(np.nanstd(l0_samples))
        m0_med = float(np.median(m0_samples))
        m0_sd = float(np.nanstd(m0_samples))
        sigma_u_med = float(np.median(sigma_u_samples))
        sigma_u_sd = float(np.nanstd(sigma_u_samples))
        ratio_med = float(np.median(ratio_samples))
        ratio_sd = float(np.nanstd(ratio_samples))
        thetav_med = float(np.median(thetav_samples))
        thetav_sd = float(np.nanstd(thetav_samples))

        rad_sigma_med_img = 1/(2*np.pi*sigma_u_med)
        sigma_med_img = Angle(rad_sigma_med_img, units.radian).to(units.arcsec).value
        rad_sigma_sd_img = 1/(2*np.pi*sigma_u_med**2) * sigma_u_sd
        sigma_sd_img = Angle(rad_sigma_sd_img, units.radian).to(units.arcsec).value

        source_nbeams = 2*np.pi * rad_sigma_med_img**2 / rad_barea
        peak_med = s0_med / source_nbeams
        peak_sd = s0_sd / source_nbeams

        theta_med_img = float((thetav_med * 180/np.pi-90)%90) # angle that major axis is rotated CCW from north
        theta_sd_img = float(thetav_sd * 180/np.pi)

        if corner_plot:
            fig = corner.corner(chain1, labels=["s0", "l0", "m0", "sigma_min", "ratio", "theta"])

        model = s0_med * np.exp(-1/2 * ((u*np.cos(thetav_med)-v*np.sin(thetav_med))**2/sigma_u_med**2 + \
                        (u*np.sin(thetav_med)+v*np.cos(thetav_med))**2/(sigma_u_med*ratio_med)**2)) \
                        * np.exp(-2*np.pi*1j*(u*l0_med + v*m0_med))
        chi2 = float(np.sum(w * ((re - model.real)**2 + (im - model.imag)**2)))

        if ratio_med > 0.85 and thetav_sd*180/np.pi > 10: # probably circular
            circular_run = circlular_gaussian_uv_fit(fits_file, re, im, u, v, w, cdelt1, cunit1, naxis1, corner_plot)
            return circular_run

        else:
            second_run = {"peak_med": float(peak_med), "peak_sd": float(peak_sd), "x_med": float(Angle(l0_med, units.radian).to(units.arcsec).value), \
                "x_sd": float(Angle(l0_sd, units.radian).to(units.arcsec).value), "y_med": float(Angle(m0_med, units.radian).to(units.arcsec).value), \
                "y_sd": float(Angle(m0_sd, units.radian).to(units.arcsec).value), "sigma_min_med": float(sigma_med_img), \
                "sigma_min_sd": float(sigma_sd_img), "sigma_maj_med": float(sigma_med_img/ratio_med), \
                "sigma_maj_sd": float(sigma_med_img/ratio_med * ratio_sd), "theta_med": theta_med_img, "theta_sd": theta_sd_img, "chi2": chi2}
            return second_run
    else:
        theta_med_img = float((thetav_med * 180/np.pi-90)%90) # angle that major axis is rotated CCW from north
        theta_sd_img = float(thetav_sd * 180/np.pi)

        model = s0_med * np.exp(-1/2 * ((u*np.cos(thetav_med)-v*np.sin(thetav_med))**2/sigma_u_med**2 + \
                        (u*np.sin(thetav_med)+v*np.cos(thetav_med))**2/(sigma_u_med*ratio_med)**2)) \
                        * np.exp(-2*np.pi*1j*(u*l0_med + v*m0_med))
        chi2 = float(np.sum(w * ((re - model.real)**2 + (im - model.imag)**2)))

        first_run = {"peak_med": float(peak_med), "peak_sd": float(peak_sd), "x_med": float(Angle(l0_med, units.radian).to(units.arcsec).value), \
                "x_sd": float(Angle(l0_sd, units.radian).to(units.arcsec).value), "y_med": float(Angle(m0_med, units.radian).to(units.arcsec).value), \
                "y_sd": float(Angle(m0_sd, units.radian).to(units.arcsec).value), "sigma_min_med": float(sigma_med_img), \
                "sigma_min_sd": float(sigma_sd_img), "sigma_maj_med": float(sigma_med_img/ratio_med), \
                "sigma_maj_sd": float(sigma_med_img/ratio_med * ratio_sd), "theta_med": theta_med_img, "theta_sd": theta_sd_img, "chi2": chi2}
        return first_run

In [ ]:
def disk_uv_fit(fits_file: str, re, im, u, v, w, cdelt1, cunit1, naxis1, corner_plot: bool = True):

    nsteps = 100
    ndim = 4
    nwalkers = 2 * ndim

    p0 = np.zeros((nwalkers, ndim))
    # summ = summary(fits_file, plot=False)
    # coord0 = summ['int_peak_coord'][0] # relative arcsec
    coord0 = (0, 0) # testing purposes
    rad_coord = (float(Angle(coord0[0], units.arcsec).to(units.radian).value), float(Angle(coord0[1], units.arcsec).to(units.radian).value))
    # bmaj = summ['fwhm'] # arcsec
    # bmin = file[0].header['BMIN'] # arcsec
    bmaj = 3.3 # arcsec, testing purposes
    bmin = 3 # arcsec, testing purposes
    rad_bmaj = Angle(bmaj, 'arcsec').to(units.radian).value
    rad_bmin = Angle(bmin, 'arcsec').to(units.radian).value
    rad_barea = np.pi * rad_bmaj * rad_bmin / (4 * np.log(2)) # because FWHM
    # img_peak = summ['int_peak_val'][0] # Jy
    img_peak = 0.08 # Jy, testing purposes
    total_flux = np.max((re**2 + im**2)**(1/2))
    rad_pix = float(Angle(cdelt1, cunit1).to(units.radian).value)
    l0_guess = rad_coord[0]
    m0_guess = rad_coord[1]
    source_area_guess = total_flux / img_peak * rad_barea
    img_radius_guess = np.sqrt(source_area_guess / np.pi)

    # print(Angle(img_radius_guess, units.radian).to(units.arcsec).value) # debug
    radius_guess = 1/(2*np.pi*img_radius_guess) # lambda

    for i in range(nwalkers):
        p0[i,0] = np.random.uniform(total_flux*0.95, total_flux*1.05)
        p0[i,1] = np.random.uniform(-rad_pix/2 + l0_guess, rad_pix/2 + l0_guess)
        p0[i,2] = np.random.uniform(-rad_pix/2 + m0_guess, rad_pix/2 + m0_guess)
        p0[i,3] = np.random.uniform(radius_guess*0.95, radius_guess*1.05)

    print(p0) # debug

    sampler = emcee.EnsembleSampler(nwalkers, ndim, d_log_probability, args=(re, im, u, v, w, rad_coord, rad_bmaj))
    try:
        state = sampler.run_mcmc(p0, nsteps)
    except emcee.autocorr.AutocorrError:
        pass
    tau = sampler.get_autocorr_time(quiet=True)
    int_tau = math.ceil(np.nanmax(tau)/1)
    steps_to_200_tau = int_tau * 200 - nsteps
    sampler.run_mcmc(state,  steps_to_200_tau)
    chain = sampler.get_chain(discard = int_tau * 10, flat=True)

    s0_samples = [i[0] for i in chain]
    l0_samples = [i[1] for i in chain]
    m0_samples = [i[2] for i in chain]
    r_samples = [i[3] for i in chain]

    s0_med = float(np.median(s0_samples))
    s0_sd = float(np.nanstd(s0_samples))
    l0_med = float(np.median(l0_samples))
    l0_sd = float(np.nanstd(l0_samples))
    m0_med = float(np.median(m0_samples))
    m0_sd = float(np.nanstd(m0_samples))
    r_med = float(np.median(r_samples))
    r_sd = float(np.nanstd(r_samples))

    if corner_plot:
        fig = corner.corner(chain, labels=["s0", "l0", "m0", "r"])

    q = (u**2 + v**2)**0.5 # in lambda
    model = s0_med * 2*r_med/q * sp.j1(q/r_med) * np.exp(-2*np.pi*1j*(u*l0_med - v*m0_med))
    chi2 = float(np.sum(w * ((re - model.real)**2 + (im - model.imag)**2)))

    if r_sd > r_med/4 or s0_sd > s0_med/4: # uncertain fit
        p1 = np.zeros((nwalkers, ndim))
        for i in range(nwalkers):
            p1[i,0] = np.random.uniform(s0_med*0.95, s0_med*1.05)
            p1[i,1] = np.random.uniform(-rad_pix/4 + l0_guess, rad_pix/4 + l0_guess)
            p1[i,2] = np.random.uniform(-rad_pix/4 + m0_guess, rad_pix/4 + m0_guess)
            p1[i,3] = np.random.uniform(r_med*0.95, r_med*1.05)

        print(p1) # debug

        sampler1 = emcee.EnsembleSampler(nwalkers, ndim, d_log_probability, args=(re, im, u, v, w, rad_coord, rad_bmaj))
        nsteps = 100 # update nsteps for second run
        try:
            state = sampler1.run_mcmc(p1, nsteps)
        except emcee.autocorr.AutocorrError:
            pass
        tau = sampler1.get_autocorr_time(quiet=True)

        int_tau = math.ceil(np.nanmax(tau)/1)
        steps_to_200_tau = int_tau * 200 - nsteps
        sampler1.run_mcmc(state,  steps_to_200_tau)
        chain1 = sampler1.get_chain(discard = int_tau * 10, flat=True)

        s0_samples = [i[0] for i in chain1]
        l0_samples = [i[1] for i in chain1]
        m0_samples = [i[2] for i in chain1]
        r_samples = [i[3] for i in chain1]

        s0_med = float(np.median(s0_samples))
        s0_sd = float(np.nanstd(s0_samples))
        l0_med = float(np.median(l0_samples))
        l0_sd = float(np.nanstd(l0_samples))
        m0_med = float(np.median(m0_samples))
        m0_sd = float(np.nanstd(m0_samples))
        r_med = float(np.median(r_samples))
        r_sd = float(np.nanstd(r_samples))

        if corner_plot:
            fig = corner.corner(chain1, labels=["s0", "l0", "m0", "r"])

        r_med_img = 1/(2*np.pi*r_med) # radians
        r_sd_img = 1/(2*np.pi*(r_med**2)) * r_sd # radians
        source_nbeams = np.pi * (r_med_img**2) / rad_barea

        peak_med = s0_med / source_nbeams
        peak_sd = s0_sd / source_nbeams

        model = s0_med * 2*r_med/q * sp.j1(q/r_med) * np.exp(-2*np.pi*1j*(u*l0_med - v*m0_med))
        chi2 = float(np.sum(w * ((re - model.real)**2 + (im - model.imag)**2)))

        second_run = {"peak_med": float(peak_med), "peak_sd": float(peak_sd), "x_med": float(Angle(l0_med, units.radian).to(units.arcsec).value), \
                "x_sd": float(Angle(l0_sd, units.radian).to(units.arcsec).value), "y_med": float(Angle(m0_med, units.radian).to(units.arcsec).value), \
                "y_sd": float(Angle(m0_sd, units.radian).to(units.arcsec).value), \
                "r_med": float(Angle(r_med_img, units.radian).to(units.arcsec).value), \
                "r_sd": float(Angle(r_sd_img, units.radian).to(units.arcsec).value), "chi2": chi2}

        return second_run
    else:
        r_med_img = 1/(2*np.pi*r_med) # radians
        r_sd_img = 1/(2*np.pi*(r_med**2)) * r_sd # radians
        source_nbeams = np.pi * (r_med_img**2) / rad_barea

        peak_med = s0_med / source_nbeams
        peak_sd = s0_sd / source_nbeams

        first_run = {"peak_med": float(peak_med), "peak_sd": float(peak_sd), "x_med": float(Angle(l0_med, units.radian).to(units.arcsec).value), \
                "x_sd": float(Angle(l0_sd, units.radian).to(units.arcsec).value), "y_med": float(Angle(m0_med, units.radian).to(units.arcsec).value), \
                "y_sd": float(Angle(m0_sd, units.radian).to(units.arcsec).value), \
                "r_med": float(Angle(r_med_img, units.radian).to(units.arcsec).value), \
                "r_sd": float(Angle(r_sd_img, units.radian).to(units.arcsec).value), "chi2": chi2}
        return first_run

In [ ]:
def double_point_uv_fit(fits_file: str, re, im, u, v, w, cdelt1, cunit1, naxis, corner_plot: bool = True):

    nsteps = 100
    ndim = 6
    nwalkers = 2 * ndim

    # initial positions for re and im
    p0 = np.zeros((nwalkers, ndim))

    rad_pix = float(Angle(cdelt1, cunit1).to(units.radian).value)
    # summ = summary(fits_file, plot=False)
    # coord1 = summ['int_peak_coord'][0] # relative arcsec
    coord1 = (0.0, 3.0) # testing purposes
    rad_coord1 = (float(Angle(coord1[0], units.arcsec).to(units.radian).value), float(Angle(coord1[1], units.arcsec).to(units.radian).value))
    bmaj = 3  # arcsec, testing purposes
    # bmaj = summ['fwhm'] # arcsec
    rad_bmaj = float(Angle(bmaj, units.arcsec).to(units.radian).value)
    # peak_guess1 = summ['int_peak_val'][0] # Jy
    peak_guess1 = 0.008 # Jy, testing purposes

    peak_guess2 = 0.006 # Jy, testing purposes
    rad_coord2 = (Angle(5.0, units.arcsec).to(units.radian).value, 0.0)
    # if len(summ['int_peak_coord']) > 1:
    #     coord2 = summ['int_peak_coord'][1] # relative arcsec
    #     rad_coord2 = (float(Angle(coord2[0], units.arcsec).to(units.radian).value), float(Angle(coord2[1], units.arcsec).to(units.radian).value))
    #     peak_guess2 = summ['int_peak_val'][1] # Jy
    # if type(summ['ext_peak_coord']) == list:
    #     if peak_guess2 < summ['ext_peak_val'][0]:
    #         coord2 = summ['ext_peak_coord'][0]
    #         rad_coord2 = (float(Angle(coord2[0], units.arcsec).to(units.radian).value), float(Angle(coord2[1], units.arcsec).to(units.radian).value))
    #         peak_guess2 = summ['ext_peak_val'][0] # Jy

    # if len(summ['int_peak_coord']) == 1 and type(summ['ext_peak_coord']) != list:
        # warnings.warn("Only one point source found in image analysis. Double point source fit may not be reliable.")
        # rad_half_axis = naxis/2 * rad_pix
        # for i in range(nwalkers):
        #     p0[i,3] = np.random.uniform(peak_guess1*0.1, peak_guess1)
        #     p0[i,4] = np.random.uniform(-rad_half_axis, rad_half_axis)
        #     p0[i,5] = np.random.uniform(-rad_half_axis, rad_half_axis)

    for i in range(nwalkers):
        p0[i,0] = np.random.uniform(0.95*peak_guess1, 1.05*peak_guess1)
        p0[i,1] = np.random.uniform(-rad_pix/2 + rad_coord1[0], rad_pix/2 + rad_coord1[0])
        p0[i,2] = np.random.uniform(-rad_pix/2 + rad_coord1[1], rad_pix/2 + rad_coord1[1])
        p0[i,3] = np.random.uniform(0.95*peak_guess2, 1.05*peak_guess2)
        p0[i,4] = np.random.uniform(-rad_pix/2 + rad_coord2[0], rad_pix/2 + rad_coord2[0])
        p0[i,5] = np.random.uniform(-rad_pix/2 + rad_coord2[1], rad_pix/2 + rad_coord2[1])

    sampler = emcee.EnsembleSampler(nwalkers, ndim, dp_log_probability, args=(re, im, u, v, w, rad_coord1, rad_coord2, rad_bmaj))
    try:
        state = sampler.run_mcmc(p0, nsteps)
    except emcee.autocorr.AutocorrError:
        pass
    tau = sampler.get_autocorr_time(quiet=True)
    int_tau = math.ceil(np.nanmax(tau)/1)
    steps_to_200_tau = int_tau * 200 - nsteps
    sampler.run_mcmc(state,  steps_to_200_tau)
    chain = sampler.get_chain(discard = int_tau * 5, flat=True)

    s1_samples = [i[0] for i in chain]
    x1_samples = [i[1] for i in chain]
    y1_samples = [i[2] for i in chain]
    s2_samples = [i[3] for i in chain]
    x2_samples = [i[4] for i in chain]
    y2_samples = [i[5] for i in chain]

    s1_med = float(np.median(s1_samples))
    s1_sd = float(np.nanstd(s1_samples))
    x1_med = float(np.median(x1_samples))
    x1_sd = float(np.nanstd(x1_samples))
    y1_med = float(np.median(y1_samples))
    y1_sd = float(np.nanstd(y1_samples))
    s2_med = float(np.median(s2_samples))
    s2_sd = float(np.nanstd(s2_samples))
    x2_med = float(np.median(x2_samples))
    x2_sd = float(np.nanstd(x2_samples))
    y2_med = float(np.median(y2_samples))
    y2_sd = float(np.nanstd(y2_samples))

    model = s1_med * np.exp(-2*np.pi*1j*(u*x1_med + v*y1_med)) + s2_med * np.exp(-2*np.pi*1j*(u*x2_med + v*y2_med))
    chi2 = float(np.sum(w * ((re - model.real)**2 + (im - model.imag)**2)))

    if corner_plot:
        fig = corner.corner(chain, labels=["s1", "l1", "m1", "s2", "l2", "m2"])

    return {"peak1_med": s1_med, "peak1_sd": s1_sd, "x1_med": float(Angle(x1_med, 'rad').to(units.arcsec).value), \
            "x1_sd": float(Angle(x1_sd, 'rad').to(units.arcsec).value), "y1_med": float(Angle(y1_med, 'rad').to(units.arcsec).value), \
            "y1_sd": float(Angle(y1_sd, 'rad').to(units.arcsec).value), "peak2_med": s2_med, "peak2_sd": s2_sd, \
            "x2_med": float(Angle(x2_med, 'rad').to(units.arcsec).value), "x2_sd": float(Angle(x2_sd, 'rad').to(units.arcsec).value), \
            "y2_med": float(Angle(y2_med, 'rad').to(units.arcsec).value), "y2_sd": float(Angle(y2_sd, 'rad').to(units.arcsec).value), "chi2": chi2}

In [679]:
def mcmc_uv_fit(fits_file: str, source_type, corner_plot: bool = True):

    if source_type not in ['p', 'cg', 'g', 'd', 'dp', 'any']:
        raise ValueError("source_type must be one of the following: 'p' (point), 'cg' (circular gaussian), 'g' (gaussian), 'd' (disk), 'dp' (double point), or 'any' (try all and pick best fit)")
    file = fits.open(fits_file)
    cdelt1 = file[0].header['CDELT1']
    cunit1 = file[0].header['CUNIT1']
    naxis1 = file[0].header['NAXIS1']
    data = file[1].data
    vis = np.array(data)
    freq_bin, u, v, re, im, w = [], [], [], [], [], []
    for row in vis:
        freq_bin_data, u_data, v_data, re_data, im_data, w_data = row
        freq_bin.append(int(freq_bin_data))
        u.append(int(u_data))
        v.append(int(v_data))
        re.append(float(re_data/w_data))
        im.append(float(im_data/w_data))
        w.append(float(w_data))

    # adding in conjugate half of data
    freq_bin *= 2
    neg_u = [-1 * val for val in u]
    u += neg_u
    neg_v = [-1 * val for val in v]
    v += neg_v
    re *= 2
    neg_im = [-1 * val for val in im]
    im += neg_im
    w *= 2

    freq_bin = np.array(freq_bin)
    u = np.array(u)
    v = np.array(v)
    re = np.array(re)
    im = np.array(im)
    w = np.array(w)

    if source_type == 'p':
        return point_uv_fit(fits_file, re, im, u, v, w, cdelt1, cunit1, corner_plot)
    if source_type == 'cg':
        return circlular_gaussian_uv_fit(fits_file, re, im, u, v, w, cdelt1, cunit1, naxis1, corner_plot)
    if source_type == 'g':
        return gaussian_uv_fit(fits_file, re, im, u, v, w, cdelt1, cunit1, naxis1, corner_plot)
    if source_type == 'd':
        return disk_uv_fit(fits_file, re, im, u, v, w, cdelt1, cunit1, naxis1, corner_plot)
    if source_type == 'dp':
        return double_point_uv_fit(fits_file, re, im, u, v, w, cdelt1, cunit1, naxis1, corner_plot)
    if source_type == 'any':
        try:
            p_dict = point_uv_fit(fits_file, re, im, u, v, w, cdelt1, cunit1, corner_plot=False)
        except:
            p_dict = {'chi2': np.inf}
        try:
            cg_dict = circlular_gaussian_uv_fit(fits_file, re, im, u, v, w, cdelt1, cunit1, naxis1, corner_plot=False)
        except:
            cg_dict = {'chi2': np.inf}
        try:
            g_dict = gaussian_uv_fit(fits_file, re, im, u, v, w, cdelt1, cunit1, naxis1, corner_plot=False)
        except:
            g_dict = {'chi2': np.inf}
        try:
            d_dict = disk_uv_fit(fits_file, re, im, u, v, w, cdelt1, cunit1, naxis1, corner_plot=False)
        except:
            d_dict = {'chi2': np.inf}
        try:
            dp_dict = double_point_uv_fit(fits_file, re, im, u, v, w, cdelt1, cunit1, naxis1, corner_plot=False)
        except:
            dp_dict = {'chi2': np.inf}

        p_chi2 = p_dict['chi2']
        cg_chi2 = cg_dict['chi2']
        g_chi2 = g_dict['chi2']
        d_chi2 = d_dict['chi2']
        dp_chi2 = dp_dict['chi2']

        p_k = 3
        cg_k = 4
        if 'theta_med' in g_dict.keys():
            g_k = 6
        else:
            g_k = 4
        d_k = 4
        dp_k = 6
        n = len(u)

        p_bic = p_chi2 + p_k * np.log(n)
        cg_bic = cg_chi2 + cg_k * np.log(n)
        g_bic = g_chi2 + g_k * np.log(n)
        d_bic = d_chi2 + d_k * np.log(n)
        dp_bic = dp_chi2 + dp_k * np.log(n)

        bic_dict = {'p': p_bic, 'cg': cg_bic, 'g': g_bic, 'd': d_bic, 'dp': dp_bic}
        print(bic_dict)
        best_fit = min(bic_dict.values())
        candidate_types = [key for key, value in bic_dict.items() if value == best_fit]

        for i in range(len(candidate_types)):
            if candidate_types[i] == 'p':
                return {'p': p_dict}
            if candidate_types[i] == 'cg':
                return {'cg': cg_dict}
            if candidate_types[i] == 'd':
                return {'d': d_dict}
            if candidate_types[i] == 'g':
                return {'g': g_dict}
            if candidate_types[i] == 'dp':
                return {'dp': dp_dict}


In [680]:
# ##### DISK TESTING #####
# r = 5.0 # arcsec
# peak = 0.08 # Jy
# fluxdensity = (np.pi*r**2)/(np.pi*3*3.3/(4*np.log(2))) * peak # Jy
# err = 0.0001 # Jy
# weight = 1/(err**2)

# pixsize = 0.2 # Satisfy Nyquist criterion, in arcsec
# imsize = 40.0 # arcsec
# npix = int(np.round(imsize/pixsize)) # Make sure this is odd?
# coords = np.arange(npix)*pixsize-npix/2.0*pixsize#+0.000001
# xcoords2d = np.reshape(np.repeat(coords, npix), (npix,npix))
# ycoords2d = xcoords2d.T
# diskimage = np.zeros((npix,npix))
# distimage = np.sqrt(xcoords2d**2.0+ycoords2d**2.0)
# diskimage[distimage <= r] = peak / ((np.pi*3*3.3/(4*np.log(2)))/pixsize**2)

# diskvisfunc=np.fft.fftshift(np.fft.fft2(np.fft.fftshift(diskimage))) # normalize
# #diskvisfunc *= fluxdensity / np.max(np.abs(diskvisfunc))
# vispixsize=1./(npix*pixsize/3600.0*np.pi/180.0) # lambda
# viscoords=np.arange(npix)*vispixsize-vispixsize*npix/2.0
# visucoords2d=np.reshape(np.repeat(viscoords, npix), (npix,npix))
# visvcoords2d=visucoords2d.T
# uvdistimage=np.sqrt(visucoords2d**2.0+visvcoords2d**2.0)

# num_vis_points = len(diskvisfunc)**2
# noise_real = np.array(np.random.normal(0, err, num_vis_points))
# noise_imag = np.array(np.random.normal(0, err, num_vis_points))

# visucoords_flat = visucoords2d.flatten()
# visvcoords_flat = visvcoords2d.flatten()
# diskvisfunc_flat = np.array(diskvisfunc.flatten())
# noisy_vis = diskvisfunc_flat + noise_real + 1j*noise_imag

# vis = []
# for i in range(num_vis_points):
#     if (np.random.random() < 0.8 and visucoords_flat[i] != 0 and visvcoords_flat[i] != 0):
#         newrow = (0, float(visucoords_flat[i]), float(visvcoords_flat[i]), float(noisy_vis[i].real*weight), float(noisy_vis[i].imag*weight), weight)
#         vis.append(newrow)

# ##### FIT TO FAKE DISK #####

# freq_bin, u, v, re, im, w = [], [], [], [], [], []
# for row in vis:
#     freq_bin_data, u_data, v_data, re_data, im_data, w_data = row
#     freq_bin.append(int(freq_bin_data))
#     u.append(float(u_data))
#     v.append(float(v_data))
#     re.append(float(re_data/w_data))
#     im.append(float(im_data/w_data))
#     w.append(float(w_data))

# freq_bin = np.array(freq_bin)
# u = np.array(u)
# v = np.array(v)
# re = np.array(re)
# im = np.array(im)
# w = np.array(w)

# # shifting disk
# shiftarcsec = 0 # arcsec
# shiftangle = 0 # degrees
# shiftuv = -2.0*np.pi*u*shiftarcsec*np.cos(shiftangle*np.pi/180.0)*np.pi/180.0/3600.0-2.0*np.pi*v*shiftarcsec*np.sin(shiftangle*np.pi/180.0)*np.pi/180.0/3600.0
# datash = (1*re+1j*im)*(1*np.cos(shiftuv)+1j*np.sin(shiftuv))
# re=datash.real
# im=datash.imag

# disk_uv_fit('', re, im, u, v, w, pixsize, 'arcsec', npix, corner_plot=True)

In [681]:
# ##### POINT TESTING #####
# fluxdensity = 0.01 # Jy
# img_err = 0.001 # Jy

# pixsize = 0.2 # Satisfy Nyquist criterion, in arcsec
# imsize = 40.0 # arcsec
# npix = int(np.round(imsize/pixsize)) # Make sure this is odd?
# coords = np.arange(npix)*pixsize-npix/2.0*pixsize
# xcoords2d = np.reshape(np.repeat(coords, npix), (npix,npix))
# ycoords2d = xcoords2d.T
# pimage = np.zeros((npix,npix))
# pimagedist = np.sqrt(xcoords2d**2.0+ycoords2d**2.0)
# pimage[npix//2, npix//2] = fluxdensity

# pvisfunc=np.fft.fftshift(np.fft.fft2(np.fft.fftshift(pimage)))

# vispixsize=1./(npix*pixsize/3600.0*np.pi/180.0) # lambda
# viscoords=np.arange(npix)*vispixsize-vispixsize*npix/2.0
# visucoords2d=np.reshape(np.repeat(viscoords, npix), (npix,npix))
# visvcoords2d=visucoords2d.T
# uvdistimage=np.sqrt(visucoords2d**2.0+visvcoords2d**2.0)

# num_vis_points = len(pvisfunc)**2
# err = img_err * np.sqrt(num_vis_points) # rough estimate of visibility noise from image noise
# noise_real = np.array(np.random.normal(0, err, num_vis_points))
# noise_imag = np.array(np.random.normal(0, err, num_vis_points))
# weights = 1/np.sqrt(noise_real**2 + noise_imag**2)

# visucoords_flat = visucoords2d.flatten()
# visvcoords_flat = visvcoords2d.flatten()
# pvisfunc_flat = np.array(pvisfunc.flatten())
# noisy_vis = pvisfunc_flat + noise_real + 1j*noise_imag

# vis = []
# for i in range(num_vis_points):
#     newrow = (0, float(visucoords_flat[i]), float(visvcoords_flat[i]), float(noisy_vis[i].real*weights[i]), float(noisy_vis[i].imag*weights[i]), weights[i])
#     vis.append(newrow)

# ##### FIT TO FAKE POINT #####

# freq_bin, u, v, re, im, w = [], [], [], [], [], []
# for row in vis:
#     freq_bin_data, u_data, v_data, re_data, im_data, w_data = row
#     freq_bin.append(int(freq_bin_data))
#     u.append(float(u_data))
#     v.append(float(v_data))
#     re.append(float(re_data/w_data))
#     im.append(float(im_data/w_data))
#     w.append(float(w_data))

# freq_bin = np.array(freq_bin)
# u = np.array(u)
# v = np.array(v)
# re = np.array(re)
# im = np.array(im)
# w = np.array(w)

# # shifting point
# shiftarcsec = 0 # arcsec
# shiftrad = Angle(shiftarcsec, 'arcsec').to(units.radian).value
# shiftangle = 0 # degrees
# shiftuv = -2.0*np.pi*u*shiftrad*np.cos(shiftangle*np.pi/180.0)-2.0*np.pi*v*shiftrad*np.sin(shiftangle*np.pi/180.0)
# datash = (1*re+1j*im)*(1*np.cos(shiftuv)+1j*np.sin(shiftuv))
# re=datash.real
# im=datash.imag

# print(np.average(w**(-1/2)))
# print(np.median(w**(-1/2)))
# print(np.average(w))
# print(np.nanstd(w))
# print(np.nanstd(np.sqrt(re**2 + im**2)))
# point_uv_fit('', re, im, u, v, w, pixsize, 'arcsec', corner_plot=True)

In [682]:
# file = fits.open('../data/uv_test/3c84.uvfits')
# known_w = file[1].data['Weight']
# known_re = file[1].data['Re']/known_w
# known_im = file[1].data['Im']/known_w
# print(np.average(known_w**(-1/2)))
# print(np.median(known_w**(-1/2)))
# print(np.average(known_w))
# print(np.nanstd(known_w))
# print(np.nanstd(np.sqrt(known_re**2 + known_im**2)))

In [683]:
# 3.035778 / np.sqrt(len(known_re))

{'amp_med': 2.4263074536467455,
 'amp_sd': 0.011833364364679877,
 'x_med': 0.00495042590331322,
 'x_sd': 0.004676320511525768,
 'y_med': -0.01127090750766012,
 'y_sd': 0.003408574858148316,
 'chi2': 7471.096603471777}

 image rms: 0.09

In [684]:
# ##### TESTING POINT SOURCE #####
# mcmc_uv_fit('../data/uv_test/3c84.uvfits', 'p')

In [685]:
# ##### TESTING CIRCULAR GAUSSIAN #####
# sd_x = 6 # arcsec
# sd_y = 6 # arcsec
# fluxdensity = 0.08 # Jy
# err = 0.001
# theta = 0 # radians

# weight = 1/err**2
# pixsize = 0.2  # Satisfy Nyquist criterion, in arcsec
# rad_pix = Angle(pixsize, 'arcsec').to(units.radian).value
# imsize = 40.0 # arcsec
# npix = int(np.round(imsize/pixsize))+1 #Make sure this is odd?

# coords = np.arange(npix)*pixsize-npix/2.0*pixsize #+0.000001
# xcoords2d = np.reshape(np.repeat(coords, npix), (npix,npix))
# ycoords2d = xcoords2d.T
# gaussimage = np.zeros((npix,npix))
# distimage = np.sqrt(xcoords2d**2.0+ycoords2d**2.0)
# gaussimage = np.exp(-0.5 * ((xcoords2d*np.cos(theta)-ycoords2d*np.sin(theta))**2 / (sd_x**2) + (xcoords2d*np.sin(theta)+ycoords2d*np.cos(theta))**2 / (sd_y**2)))
# gaussimage *= fluxdensity/np.sum(gaussimage)

# gaussvisfunc=np.fft.fftshift(np.fft.fft2(np.fft.fftshift(gaussimage)))
# vispixsize=1./(npix*pixsize/3600.0*np.pi/180.0)/1e3 #klambda
# viscoords=np.arange(npix)*vispixsize-vispixsize*npix/2.0
# visucoords2d=np.reshape(np.repeat(viscoords, npix), (npix,npix))
# visvcoords2d=visucoords2d.T
# uvdistimage=np.sqrt(visucoords2d**2.0+visvcoords2d**2.0)

# num_vis_points = len(gaussvisfunc)**2
# noise_real = np.array(np.random.normal(0, err, num_vis_points))
# noise_imag = np.array(np.random.normal(0, err, num_vis_points))

# visucoords_flat = visucoords2d.flatten()
# visvcoords_flat = visvcoords2d.flatten()
# gaussvisfunc_flat = np.array(gaussvisfunc.flatten())
# noisy_vis = gaussvisfunc_flat + noise_real + 1j*noise_imag

# vis = []
# for i in range(num_vis_points):
#     if np.random.random() < 0.8:
#         newrow = (0, float(visucoords_flat[i]), float(visvcoords_flat[i]), float(noisy_vis[i].real*weight), float(noisy_vis[i].imag*weight), weight)
#         vis.append(newrow)

# ##### FIT TO FAKE GAUSSIAN #####

# freq_bin, u, v, re, im, w = [], [], [], [], [], []
# for row in vis:
#     freq_bin_data, u_data, v_data, re_data, im_data, w_data = row
#     freq_bin.append(int(freq_bin_data))
#     u.append(float(u_data))
#     v.append(float(v_data))
#     re.append(float(re_data/w_data))
#     im.append(float(im_data/w_data))
#     w.append(float(w_data))

# freq_bin = np.array(freq_bin)
# u = np.array(u)
# v = np.array(v)
# re = np.array(re)
# im = np.array(im)
# w = np.array(w)

# # shifting Gaussian
# shiftarcsec = 0 #arcsec
# shiftangle = 0 #degrees
# shiftuv = -2.0*np.pi*u*shiftarcsec*np.cos(shiftangle*np.pi/180.0)*np.pi/180.0/3600.0-2.0*np.pi*v*shiftarcsec*np.sin(shiftangle*np.pi/180.0)*np.pi/180.0/3600.0
# datash = (1*re+1j*im)*(1*np.cos(shiftuv)+1j*np.sin(shiftuv))
# re=datash.real
# im=datash.imag

# circlular_gaussian_uv_fit('fake_gaussian.uvfits', re, im, u, v, w, cdelt1=pixsize, cunit1='arcsec', naxis1=npix, corner_plot=True)

In [686]:
# ##### TESTING ELLIPTICAL GAUSSIAN #####
# sd_x = 1 # arcsec
# sd_y = 3 # arcsec
# fluxdensity = 0.08 # Jy
# err = 0.001
# theta = -np.pi/2 # radians

# weight = 1/err**2
# pixsize = 0.2  # Satisfy Nyquist criterion, in arcsec
# rad_pix = Angle(pixsize, 'arcsec').to(units.radian).value
# imsize = 40.0 # arcsec
# npix = int(np.round(imsize/pixsize))+1 #Make sure this is odd?

# coords = np.arange(npix)*pixsize-npix/2.0*pixsize #+0.000001
# xcoords2d = np.reshape(np.repeat(coords, npix), (npix,npix))
# ycoords2d = xcoords2d.T
# gaussimage = np.zeros((npix,npix))
# distimage = np.sqrt(xcoords2d**2.0+ycoords2d**2.0)
# gaussimage = np.exp(-0.5 * ((xcoords2d*np.cos(theta)-ycoords2d*np.sin(theta))**2 / (sd_x**2) + (xcoords2d*np.sin(theta)+ycoords2d*np.cos(theta))**2 / (sd_y**2)))
# gaussimage *= fluxdensity/np.sum(gaussimage)

# gaussvisfunc=np.fft.fftshift(np.fft.fft2(np.fft.fftshift(gaussimage)))
# vispixsize=1./(npix*pixsize/3600.0*np.pi/180.0)/1e3 #klambda
# viscoords=np.arange(npix)*vispixsize-vispixsize*npix/2.0
# visucoords2d=np.reshape(np.repeat(viscoords, npix), (npix,npix))
# visvcoords2d=visucoords2d.T
# uvdistimage=np.sqrt(visucoords2d**2.0+visvcoords2d**2.0)

# num_vis_points = len(gaussvisfunc)**2
# noise_real = np.array(np.random.normal(0, err, num_vis_points))
# noise_imag = np.array(np.random.normal(0, err, num_vis_points))

# visucoords_flat = visucoords2d.flatten()
# visvcoords_flat = visvcoords2d.flatten()
# gaussvisfunc_flat = np.array(gaussvisfunc.flatten())
# noisy_vis = gaussvisfunc_flat + noise_real + 1j*noise_imag

# vis = []
# for i in range(num_vis_points):
#     if np.random.random() < 0.5:
#         newrow = (0, float(visucoords_flat[i]), float(visvcoords_flat[i]), float(noisy_vis[i].real*weight), float(noisy_vis[i].imag*weight), weight)
#         vis.append(newrow)

# ##### FIT TO FAKE GAUSSIAN #####

# freq_bin, u, v, re, im, w = [], [], [], [], [], []
# for row in vis:
#     freq_bin_data, u_data, v_data, re_data, im_data, w_data = row
#     freq_bin.append(int(freq_bin_data))
#     u.append(float(u_data))
#     v.append(float(v_data))
#     re.append(float(re_data/w_data))
#     im.append(float(im_data/w_data))
#     w.append(float(w_data))

# freq_bin = np.array(freq_bin)
# u = np.array(u)
# v = np.array(v)
# re = np.array(re)
# im = np.array(im)
# w = np.array(w)

# # shifting Gaussian
# shiftarcsec = 0 #arcsec
# shiftangle = 0 #degrees
# shiftuv = -2.0*np.pi*u*shiftarcsec*np.cos(shiftangle*np.pi/180.0)*np.pi/180.0/3600.0-2.0*np.pi*v*shiftarcsec*np.sin(shiftangle*np.pi/180.0)*np.pi/180.0/3600.0
# datash = (1*re+1j*im)*(1*np.cos(shiftuv)+1j*np.sin(shiftuv))
# re=datash.real
# im=datash.imag
# gaussian_uv_fit('fake_gaussian.uvfits', re, im, u, v, w, cdelt1=pixsize, cunit1='arcsec', naxis1=npix, corner_plot=True)

In [687]:
# ##### DOUBLE POINT TESTING #####
# fluxdensity1 = 0.008 # Jy
# fluxdensity2 = 0.006 # Jy
# err = 0.00001 # Jy
# weight = 1/(err**2)

# pixsize = 0.2 # Satisfy Nyquist criterion, in arcsec
# imsize = 40.0 # arcsec
# npix = int(np.round(imsize/pixsize)) # Make sure this is odd?
# coords = np.arange(npix)*pixsize-npix/2.0*pixsize#+0.000001
# xcoords2d = np.reshape(np.repeat(coords, npix), (npix,npix))
# ycoords2d = xcoords2d.T
# dpimage1 = np.zeros((npix,npix))
# dpimagedist1 = np.sqrt(xcoords2d**2.0+ycoords2d**2.0)
# dpimage1[npix //2, npix //2] = fluxdensity1
# dpimage2 = np.zeros((npix,npix))
# dpimagedist2 = np.sqrt((xcoords2d-5.0)**2.0+ycoords2d**2.0)
# dpimage2[npix //2, npix //2] = fluxdensity2

# dpvisfunc1=np.fft.fftshift(np.fft.fft2(np.fft.fftshift(dpimage1)))
# dpvisfunc2=np.fft.fftshift(np.fft.fft2(np.fft.fftshift(dpimage2)))

# vispixsize=1./(npix*pixsize/3600.0*np.pi/180.0) # lambda
# viscoords=np.arange(npix)*vispixsize-vispixsize*npix/2.0
# visucoords2d=np.reshape(np.repeat(viscoords, npix), (npix,npix))
# visvcoords2d=visucoords2d.T
# uvdistimage=np.sqrt(visucoords2d**2.0+visvcoords2d**2.0)

# num_vis_points = len(dpvisfunc1)**2
# noise_real = np.array(np.random.normal(0, err/np.sqrt(2), num_vis_points))
# noise_imag = np.array(np.random.normal(0, err/np.sqrt(2), num_vis_points))

# visucoords_flat = visucoords2d.flatten()
# visvcoords_flat = visvcoords2d.flatten()
# dpvisfunc_flat1 = np.array(dpvisfunc1.flatten())
# dpvisfunc_flat2 = np.array(dpvisfunc2.flatten())
# noisy_vis1 = dpvisfunc_flat1 + noise_real + 1j*noise_imag
# noisy_vis2 = dpvisfunc_flat2 + noise_real + 1j*noise_imag

# vis1 = []
# for i in range(num_vis_points):
#     newrow = (0, float(visucoords_flat[i]), float(visvcoords_flat[i]), float(noisy_vis1[i].real*weight), float(noisy_vis1[i].imag*weight), weight)
#     vis1.append(newrow)

# vis2 = []
# for i in range(num_vis_points):
#     newrow = (0, float(visucoords_flat[i]), float(visvcoords_flat[i]), float(noisy_vis2[i].real*weight), float(noisy_vis2[i].imag*weight), weight)
#     vis2.append(newrow)

# ##### FIT TO FAKE DOUBLE POINT #####

# freq_bin1, u1, v1, re1, im1, w1 = [], [], [], [], [], []
# for row in vis1:
#     freq_bin_data, u_data, v_data, re_data, im_data, w_data = row
#     freq_bin1.append(int(freq_bin_data))
#     u1.append(float(u_data))
#     v1.append(float(v_data))
#     re1.append(float(re_data/w_data))
#     im1.append(float(im_data/w_data))
#     w1.append(float(w_data))

# freq_bin1 = np.array(freq_bin1)
# u1 = np.array(u1)
# v1 = np.array(v1)
# re1 = np.array(re1)
# im1 = np.array(im1)
# w1 = np.array(w1)

# freq_bin2, u2, v2, re2, im2, w2 = [], [], [], [], [], []
# for row in vis2:
#     freq_bin_data, u_data, v_data, re_data, im_data, w_data = row
#     freq_bin2.append(int(freq_bin_data))
#     u2.append(float(u_data))
#     v2.append(float(v_data))
#     re2.append(float(re_data/w_data))
#     im2.append(float(im_data/w_data))
#     w2.append(float(w_data))

# freq_bin2 = np.array(freq_bin2)
# u2 = np.array(u2)
# v2 = np.array(v2)
# re2 = np.array(re2)
# im2 = np.array(im2)
# w2 = np.array(w2)

# # shifting points
# shiftarcsec1 = 3 # arcsec
# shiftangle1 = 90 # degrees
# shiftuv1 = -2.0*np.pi*u1*shiftarcsec1*np.cos(shiftangle1*np.pi/180.0)*np.pi/180.0/3600.0-2.0*np.pi*v1*shiftarcsec1*np.sin(shiftangle1*np.pi/180.0)*np.pi/180.0/3600.0
# datash1 = (1*re1+1j*im1)*(1*np.cos(shiftuv1)+1j*np.sin(shiftuv1))
# re1=datash1.real
# im1=datash1.imag

# shiftarcsec2 = 5 # arcsec
# shiftangle2 = 0 # degrees
# shiftuv2 = -2.0*np.pi*u2*shiftarcsec2*np.cos(shiftangle2*np.pi/180.0)*np.pi/180.0/3600.0-2.0*np.pi*v2*shiftarcsec2*np.sin(shiftangle2*np.pi/180.0)*np.pi/180.0/3600.0
# datash2 = (1*re2+1j*im2)*(1*np.cos(shiftuv2)+1j*np.sin(shiftuv2))
# re2=datash2.real
# im2=datash2.imag

# double_point_uv_fit('', re1+re2, im1+im2, u1, v1, w1, pixsize, 'arcsec', npix, corner_plot=True)

In [688]:
# mcmc_uv_fit('../data/uv_test/gaussian.uvfits', 'd', corner_plot=True)

In [689]:
# mcmc_uv_fit('../data/uv_test/gaussian.uvfits', 'cg', corner_plot=True)

In [690]:
# mcmc_uv_fit('../data/uv_test/gaussian.uvfits', 'g')

In [691]:
# ##### KEEPING FOR REFERENCE #####
# def cg_log_prior(params, rad_coord0, rad_bmaj):
#     s0, l0, m0, sigma_u = params
#     l0_off = abs(l0 - rad_coord0[0])
#     m0_off = abs(m0 - rad_coord0[1])
#     if 0 < s0 and 0 < sigma_u: #and l0_off < 2*rad_bmaj and m0_off < 2*rad_bmaj:
#         return 0.0
#     return -np.inf

In [692]:
# p0 = np.zeros((nwalkers, ndim))
# # summ = summary(fits_file, plot=False)
# # coord0 = summ['int_peak_coord'][0] # relative arcsec
# coord0 = (-1.0, -1.0) # testing purposes
# rad_coord = (float(Angle(coord0[0], units.arcsec).to(units.radian).value), float(Angle(coord0[1], units.arcsec).to(units.radian).value))
# # bmaj = summ['fwhm'] # arcsec
# # bmin = file[0].header['BMIN'] # arcsec
# bmaj = 3.3 # arcsec, testing purposes
# bmin = 3 # arcsec, testing purposes
# rad_bmaj = Angle(bmaj, 'arcsec').to(units.radian).value
# rad_bmin = Angle(bmin, 'arcsec').to(units.radian).value
# rad_barea = np.pi * rad_bmaj * rad_bmin / (4 * np.log(2)) # because FWHM
# # img_peak = summ['int_peak_val'][0] # Jy
# img_peak = 2.4 # Jy, testing purposes
# total_flux = np.max((re**2 + im**2)**(1/2))
# flux_guess = total_flux # testing purposes
# rad_pix = float(Angle(cdelt1, cunit1).to(units.radian).value)
# l0_guess = rad_coord[0]
# m0_guess = rad_coord[1]
# source_area_guess = total_flux / img_peak * rad_barea
# img_radius_guess = np.sqrt(source_area_guess / (2 * np.pi))
# radius_guess = 1/(2*np.pi*img_radius_guess) # lambda


In [693]:
# rad_sigma_med_img = 1/(2*np.pi*sigma_med)
# sigma_med_img = Angle(rad_sigma_med_img, units.radian).to(units.arcsec).value
# rad_sigma_sd_img = 1/(2*np.pi*sigma_med**2) * sigma_sd
# sigma_sd_img = Angle(rad_sigma_sd_img, units.radian).to(units.arcsec).value

# source_nbeams = 2*np.pi * rad_sigma_med_img**2 / rad_barea
# amp_med = s0_med / source_nbeams
# amp_sd = s0_sd / source_nbeams